# Gene accession finding from GNN clustering output

This notebook selects representative leaves from `gnn_clustering_report_*_details.csv`, resolves required identifiers via NCBI-first APIs (with UniProt fallback), and writes a plotting-compatible CSV.

## Workflow
1. Configure paths and filtering parameters
2. Load clustering details and select one representative per cluster
3. Resolve `Genome Accession` and `Taxon ID`
4. Build template-compatible output rows
5. Optionally append to existing `enzyme_ids.csv` and save

In [1]:
import os
import re
import ast
import time
from dataclasses import dataclass
from functools import lru_cache
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests
from Bio import Entrez, SeqIO


# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
BASE_DIR = r"D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search"
CLUSTERING_CSV_PATH = os.path.join(
    BASE_DIR,
    "gnn_cluster_plots_circular",
    #"gnn_clustering_report_ssn_differentiated_details.csv",
    "gnn_clustering_report_all_neighborhoods_details.csv",
)
TEMPLATE_CSV_PATH = os.path.join(BASE_DIR, "enzyme_ids.csv")
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "enzyme_ids_augmented_from_gnn.csv")

# CSV dialect controls (enzyme_ids.csv is ';' separated in your workflow)
OUTPUT_CSV_SEP = ";"

# Selection controls
SSN_FILTER: Any = "ALL"  # "ALL", single value (e.g. 39061, or 1), or list (e.g. [39061, 12345], [1, 2, 3], etc.)
USE_MEDIAN_SILHOUETTE_AS_MIN = True
SILHOUETTE_MIN_OVERRIDE: Optional[float] = None
ALWAYS_SELECT_FURTHEST_OUTWARD_LEAF = True  # if True, skip silhouette filtering and pick max radial_distance per cluster
TAKE_FURTHEST_PER_CLUSTER = True  # select max radial_distance per cluster_id
KEEP_ORIGINAL_INPUT_ALWAYS = True

# Merge/append controls
APPEND_TO_TEMPLATE = True
DEDUPE_KEY_COLUMNS = ["Genome Accession", "Locus Tag"]

# API controls (NCBI first, UniProt fallback)
ENTREZ_EMAIL = "phitro@bu.edu"
ENTREZ_API_KEY: Optional[str] = None
NCBI_MAX_RETRIES = 3
NCBI_BACKOFF_BASE = 1.5
NCBI_SLEEP_SEC = 0.11
UNIPROT_TIMEOUT_SEC = 8
UNIPROT_SLEEP_SEC = 0.08

# Performance + confidence controls - three resolution modes
# "fast": Early stop after direct/protein/proteome hits (score >= 12.0); ~30-45 sec
# "standard": Current staged approach with all fallbacks (score >= 8.0); ~50-90 sec
# "depth": All of standard + ORF-name-based NCBI nucleotide search; ~120-180 sec
RESOLUTION_MODE = "standard"  # "fast", "standard", or "depth"
MAX_NUCCORE_CANDIDATES = 45
MAX_CANDIDATES_PER_STAGE = 18
EARLY_STOP_SCORE_FAST = 12.0
EARLY_STOP_SCORE_STANDARD = 10.5
EARLY_STOP_SCORE_DEPTH = 9.0
ACCEPT_MIN_SCORE = 8.0
MIN_RECORD_BP_FOR_EARLY_STOP = 50000
ENABLE_VERBOSE_WARNINGS = False

# Output schema requirements for downstream plotting
REQUIRED_OUTPUT_COLUMNS = ["Locus Tag", "Genome Accession", "Organism", "Taxon ID"]

if not ENTREZ_EMAIL or ENTREZ_EMAIL == "your_email@example.com":
    print("[WARN] Please set ENTREZ_EMAIL before running API calls.")

Entrez.email = ENTREZ_EMAIL
if ENTREZ_API_KEY:
    Entrez.api_key = ENTREZ_API_KEY

print("Configured clustering CSV:", CLUSTERING_CSV_PATH)
print("Configured template CSV:", TEMPLATE_CSV_PATH)
print("Configured output CSV:", OUTPUT_CSV_PATH)
print(f"RESOLUTION_MODE: {RESOLUTION_MODE} (fast/standard/depth) | MAX_NUCCORE_CANDIDATES: {MAX_NUCCORE_CANDIDATES}")

Configured clustering CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\gnn_cluster_plots_circular\gnn_clustering_report_all_neighborhoods_details.csv
Configured template CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\enzyme_ids.csv
Configured output CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\enzyme_ids_augmented_from_gnn.csv
RESOLUTION_MODE: standard (fast/standard/depth) | MAX_NUCCORE_CANDIDATES: 45


In [2]:
# ==============================================================
# Utilities: parsing, filtering, representative selection, merging
# ==============================================================


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).replace("\ufeff", "").strip() for c in out.columns]
    return out


def _read_csv_flexible(path: str) -> pd.DataFrame:
    """Auto-detect CSV delimiter (',' or ';')."""
    return _normalize_columns(pd.read_csv(path, sep=None, engine="python"))


def _to_numeric(df: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def _parse_ssn_filter(value: Any) -> Optional[set]:
    """Parse SSN filter value (string, int, list, or 'ALL')."""
    if value is None:
        return None
    if isinstance(value, str):
        v = value.strip()
        if v.upper() == "ALL" or v == "":
            return None
        try:
            lit = ast.literal_eval(v)
            if isinstance(lit, (list, tuple, set)):
                return set(str(x) for x in lit)
            return {str(lit)}
        except Exception:
            return set(part.strip() for part in v.split(",") if part.strip())
    if isinstance(value, (list, tuple, set)):
        return set(str(x) for x in value)
    return {str(value)}


def _first_nonempty(*vals) -> Optional[str]:
    """Return first non-empty value from a sequence."""
    for v in vals:
        if v is None:
            continue
        try:
            if isinstance(v, float) and pd.isna(v):
                continue
        except Exception:
            pass
        s = str(v).strip()
        if s:
            return s
    return None


def _ensure_columns(df: pd.DataFrame, required_cols: Sequence[str]) -> pd.DataFrame:
    """Add missing columns while preserving all existing columns and order."""
    out = df.copy()
    for c in required_cols:
        if c not in out.columns:
            out[c] = None
    # Keep existing columns first, then ensure required columns are present.
    ordered_cols = list(out.columns)
    for c in required_cols:
        if c not in ordered_cols:
            ordered_cols.append(c)
    return out[ordered_cols]


def _rank_rows_for_dedupe(df: pd.DataFrame) -> pd.DataFrame:
    """Add ranking columns for deduplication: curated > locus_found > reviewed > score > bp_length."""
    out = df.copy()
    
    # Curated flag
    out["__rank_curated"] = out.get("Curated Match Used", pd.Series([], dtype=str)).astype(str).str.lower().eq("true").astype(int)
    
    # Locus found flag
    out["__rank_locus"] = out.get("Locus Found In Record", pd.Series([], dtype=str)).astype(str).str.lower().eq("true").astype(int)
    
    # Review needed flag (invert: rows that don't need review rank higher)
    out["__rank_review"] = (~out.get("Review Needed", pd.Series([], dtype=str)).astype(str).str.lower().eq("true")).astype(int)
    
    # Resolution score (numeric)
    out["__rank_score"] = pd.to_numeric(out.get("Resolution Score", pd.Series([], dtype=float)), errors="coerce").fillna(0)
    
    # BP length (numeric)
    out["__rank_bp"] = pd.to_numeric(out.get("Record BP Length", pd.Series([], dtype=int)), errors="coerce").fillna(0)
    
    return out


def load_and_filter_clustering_details(
    csv_path: str,
    ssn_filter: Any = "ALL",
    use_median_silhouette_as_min: bool = True,
    silhouette_min_override: Optional[float] = None,
    always_select_furthest_outward_leaf: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Load clustering CSV and apply quality filters."""
    df = _read_csv_flexible(csv_path)

    required = ["ssn_id", "cluster_id", "radial_rank_outward", "radial_distance", "hit_id", "organism", "accession_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required clustering columns: {missing}")

    df = _to_numeric(df, ["ssn_id", "cluster_id", "radial_rank_outward", "radial_distance", "silhouette_like_score"])

    allowed_ssn = _parse_ssn_filter(ssn_filter)
    before_ssn = len(df)
    if allowed_ssn is not None:
        df = df[df["ssn_id"].astype("Int64").astype(str).isin(allowed_ssn)].copy()

    threshold_used = np.nan
    before_quality = len(df)
    if always_select_furthest_outward_leaf:
        stats = {
            "rows_before_ssn_filter": before_ssn,
            "rows_after_ssn_filter": before_quality,
            "rows_after_quality_filter": len(df),
            "silhouette_threshold_used": threshold_used,
            "quality_filter_skipped_for_furthest_selection": True,
        }
        return df, stats

    if "silhouette_like_score" in df.columns and len(df) > 0:
        if silhouette_min_override is not None:
            threshold_used = float(silhouette_min_override)
        elif use_median_silhouette_as_min:
            threshold_used = float(df["silhouette_like_score"].median(skipna=True))

        if pd.notna(threshold_used):
            keep_mask = df["silhouette_like_score"].fillna(-np.inf) >= threshold_used
            if "is_original_input" in df.columns:
                keep_mask = keep_mask | df["is_original_input"].astype(str).str.lower().eq("true")
            df = df[keep_mask].copy()

    stats = {
        "rows_before_ssn_filter": before_ssn,
        "rows_after_ssn_filter": before_quality,
        "rows_after_quality_filter": len(df),
        "silhouette_threshold_used": threshold_used,
        "quality_filter_skipped_for_furthest_selection": False,
    }
    return df, stats


def select_representatives_per_cluster(
    df: pd.DataFrame,
    take_furthest_per_cluster: bool = True,
    keep_original_input_always: bool = True,
) -> pd.DataFrame:
    """Select exactly one representative per cluster_id by maximum radial_distance."""
    if df.empty:
        return df.copy()

    # Strict outward-leaf selection: group only by cluster_id and pick max radial_distance.
    df_sorted = df.sort_values(
        by=["cluster_id", "radial_distance", "radial_rank_outward", "ssn_id", "neighborhood_label", "organism", "hit_id"],
        ascending=[True, False, True, True, True, True, True],
        na_position="last",
    ).copy()

    selected_rows = []
    for _, grp in df_sorted.groupby(["cluster_id"], dropna=False):
        g = grp
        if "radial_distance" in g.columns and g["radial_distance"].notna().any():
            selected_rows.append(g.sort_values("radial_distance", ascending=False, na_position="last").iloc[0])
        else:
            selected_rows.append(g.iloc[0])

    out = pd.DataFrame(selected_rows).reset_index(drop=True)
    return out


# Execute loading and filtering
filtered_df, selection_stats = load_and_filter_clustering_details(
    CLUSTERING_CSV_PATH,
    ssn_filter=SSN_FILTER,
    use_median_silhouette_as_min=USE_MEDIAN_SILHOUETTE_AS_MIN,
    silhouette_min_override=SILHOUETTE_MIN_OVERRIDE,
    always_select_furthest_outward_leaf=ALWAYS_SELECT_FURTHEST_OUTWARD_LEAF,
)

selected_df = select_representatives_per_cluster(
    filtered_df,
    take_furthest_per_cluster=TAKE_FURTHEST_PER_CLUSTER,
    keep_original_input_always=KEEP_ORIGINAL_INPUT_ALWAYS,
)

print("Selection stats:")
for k, v in selection_stats.items():
    print(f"  {k}: {v}")
print(f"Selected representatives: {len(selected_df)}")
selected_df.head(8)


Selection stats:
  rows_before_ssn_filter: 37
  rows_after_ssn_filter: 37
  rows_after_quality_filter: 37
  silhouette_threshold_used: nan
  quality_filter_skipped_for_furthest_selection: True
Selected representatives: 6


,ssn_id,neighborhood_label,organism,hit_id,accession_id,cluster_id,cluster_size,clustering_method,distance_threshold_used,silhouette_like_score,radial_distance,angular_degree,radial_rank_outward,is_original_input,collapsed_count,collapsed_code
0,1,Thiohalomonas denitrificans._FMWD01000006,Thiohalomonas denitrificans.,FMWD01000006,A0A1G5QK56,1,31,combined,NaN,0.046614,0.624805,113.490997,7,False,1,NaN
1,1,Anaeromyxobacter diazotrophicus._BJTG01000010,Anaeromyxobacter diazotrophicus.,BJTG01000010,A0A7I9VRP5,2,1,combined,NaN,1.000000,0.674631,64.552877,5,False,1,NaN
2,1,bacterium BMS3Abin03._BDSV01000107,bacterium BMS3Abin03.,BDSV01000107,A0A2H6ED39,3,1,combined,NaN,1.000000,0.656192,35.319142,6,False,1,NaN
3,1,Thioflavicoccus mobilis 8321._CP003051,Thioflavicoccus mobilis 8321.,CP003051,L0GVW7,4,1,combined,NaN,1.000000,0.724388,2.380353,3,False,1,NaN
4,1,Candidatus Methanophaga sp. ANME-1 ERB7._MT631715,Candidatus Methanophaga sp. ANME-1 ERB7.,MT631715,A0A7G9ZD04,5,2,combined,NaN,0.081975,0.895828,329.718020,2,False,1,NaN
5,3,Thermoprotei archaeon._DSCF01000035,Thermoprotei archaeon.,DSCF01000035,A0A7J2SZI2,6,1,combined,NaN,1.000000,1.000000,320.891145,1,False,1,NaN


In [3]:
# ==============================================================
# Core helpers + base resolver (must be defined before wrappers)
# ==============================================================

# Simple caches to reduce duplicate API calls
_cache_entrez_esearch: Dict[str, Dict[str, Any]] = {}
_cache_nuccore_records: Dict[str, Any] = {}
_cache_uniprot_search: Dict[str, Dict[str, Any]] = {}

# Mode-dependent early stop threshold
if str(RESOLUTION_MODE).strip().lower() == "fast":
    EARLY_STOP_SCORE = EARLY_STOP_SCORE_FAST
elif str(RESOLUTION_MODE).strip().lower() in {"depth", "deep", "thorough"}:
    EARLY_STOP_SCORE = EARLY_STOP_SCORE_DEPTH
else:
    EARLY_STOP_SCORE = EARLY_STOP_SCORE_STANDARD


# ====================
# Regex Patterns
# ====================
_NUCCORE_PATTERN = re.compile(
    r"^(?:"
    r"(?:NC|NZ|CM|CP|AP|AC|NG|NT|NW)_?\d+(?:\.\d+)?|"  # RefSeq-like nucleotide
    r"[A-Z]{4,6}\d+(?:\.\d+)?|"  # EMBL/WGS-like nucleotide
    r"[A-Z]{2}\d{6,8}(?:\.\d+)?|"  # Short EMBL style
    r"J[A-Z]\d{6}(?:\.\d+)?"  # DDBJ style
    r")$",
    re.IGNORECASE,
)
_ASSEMBLY_PATTERN = re.compile(r"^GC[AF]_\d+(\.\d+)?$", re.IGNORECASE)
_UNIPROT_ACCESSION_PATTERN = re.compile(
    r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z0-9]{3}[0-9]){1,2})$",
    re.IGNORECASE,
)


def _warn(msg: str) -> None:
    if ENABLE_VERBOSE_WARNINGS:
        print(msg)


def _is_nuccore_like(value: Optional[str]) -> bool:
    if not value:
        return False
    return bool(_NUCCORE_PATTERN.match(str(value).strip()))


def _is_assembly_like(value: Optional[str]) -> bool:
    if not value:
        return False
    return bool(_ASSEMBLY_PATTERN.match(str(value).strip()))


def _looks_uniprot_accession(value: Optional[str]) -> bool:
    if not value:
        return False
    return bool(_UNIPROT_ACCESSION_PATTERN.match(str(value).strip()))


def _is_locus_like_tag(value: Optional[str]) -> bool:
    if not value:
        return False
    return bool(re.match(r"^[A-Za-z][A-Za-z0-9]{1,24}_[A-Za-z0-9-]{2,}$", str(value).strip()))


def _dedupe_keep_order(values: Sequence[Any]) -> List[str]:
    out: List[str] = []
    seen = set()
    for v in values:
        s = str(v).strip() if v is not None else ""
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out


def _nuccore_variants(value: Optional[str]) -> List[str]:
    """Generate accession variants (e.g., add .1 for unversioned hit_id)."""
    v = str(value or "").strip()
    if not v:
        return []
    variants = [v]
    if _is_nuccore_like(v) and "." not in v:
        variants.append(f"{v}.1")
    return _dedupe_keep_order(variants)


def _safe_entrez_esearch(db: str, term: str, retmax: int = 20) -> Dict[str, Any]:
    """Safe Entrez esearch wrapper with retries and small cache."""
    cache_key = f"{db}|{term}|{retmax}"
    if cache_key in _cache_entrez_esearch:
        return _cache_entrez_esearch[cache_key]

    delay = NCBI_BACKOFF_BASE
    for attempt in range(NCBI_MAX_RETRIES):
        try:
            time.sleep(NCBI_SLEEP_SEC)
            handle = Entrez.esearch(db=db, term=term, retmax=retmax)
            result = Entrez.read(handle)
            handle.close()
            out = {"IdList": list(result.get("IdList", []))}
            _cache_entrez_esearch[cache_key] = out
            return out
        except Exception as exc:
            if attempt == NCBI_MAX_RETRIES - 1:
                _warn(f"[WARN] esearch failed for {db}:{term} -> {exc}")
                return {"IdList": []}
            time.sleep(delay)
            delay *= 1.8

    return {"IdList": []}


def _safe_entrez_esummary(db: str, ids: str) -> Dict[str, Any]:
    """Safe Entrez esummary wrapper."""
    delay = NCBI_BACKOFF_BASE
    for attempt in range(NCBI_MAX_RETRIES):
        try:
            time.sleep(NCBI_SLEEP_SEC)
            handle = Entrez.esummary(db=db, id=ids)
            result = Entrez.read(handle)
            handle.close()
            return result
        except Exception as exc:
            if attempt == NCBI_MAX_RETRIES - 1:
                _warn(f"[WARN] esummary failed for {db}:{ids} -> {exc}")
                return {}
            time.sleep(delay)
            delay *= 1.8
    return {}


def _safe_requests_json(url: str, params: Dict[str, Any], timeout: int = 8) -> Dict[str, Any]:
    """HTTP GET with retries and JSON parsing."""
    delay = 1.0
    for attempt in range(3):
        try:
            time.sleep(UNIPROT_SLEEP_SEC)
            resp = requests.get(url, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            if attempt == 2:
                _warn(f"[WARN] HTTP JSON failed for {url}: {exc}")
                return {}
            time.sleep(delay)
            delay *= 1.7
    return {}


def _safe_uniprot_search(token: str) -> Dict[str, Any]:
    """Search UniProt by accession or cross-reference token."""
    t = str(token or "").strip()
    if not t:
        return {}
    if t in _cache_uniprot_search:
        return _cache_uniprot_search[t]

    url = "https://rest.uniprot.org/uniprotkb/search"
    query = f"(accession:{t}) OR (xref:{t})"
    params = {
        "query": query,
        "format": "json",
        "size": 1,
        "fields": ",".join(
            [
                "accession",
                "organism_name",
                "organism_id",
                "gene_names",
                "xref_refseq",
                "xref_embl",
                "xref_genbank",
                "xref_bioproject",
                "xref_proteomes",
                "xref_assembly",
                "sequence",
            ]
        ),
    }
    data = _safe_requests_json(url, params, timeout=UNIPROT_TIMEOUT_SEC)
    _cache_uniprot_search[t] = data
    return data


def _extract_uniprot_bundle(entry: Dict[str, Any], query_token: str) -> Dict[str, Any]:
    """Extract relevant resolution hints from a UniProt record."""
    organism_block = entry.get("organism", {}) or {}
    genes = entry.get("genes", []) or []
    xrefs = entry.get("uniProtKBCrossReferences", []) or []

    gene_names: List[str] = []
    orf_names: List[str] = []
    refseq_ids: List[str] = []
    embl_ids: List[str] = []
    genbank_ids: List[str] = []
    proteome_ids: List[str] = []
    bioproject_ids: List[str] = []
    assembly_ids: List[str] = []
    protein_ids: List[str] = []

    for g in genes:
        gname = ((g or {}).get("geneName") or {}).get("value")
        if gname:
            gene_names.append(str(gname).strip())
        for syn in (g or {}).get("synonyms", []) or []:
            val = (syn or {}).get("value")
            if val:
                gene_names.append(str(val).strip())
        for orf in (g or {}).get("orfNames", []) or []:
            val = (orf or {}).get("value")
            if val:
                orf_names.append(str(val).strip())

    for xr in xrefs:
        db = str((xr or {}).get("database", "")).strip().lower()
        xid = str((xr or {}).get("id", "")).strip()
        if not xid:
            continue

        props = (xr or {}).get("properties", []) or []
        for p in props:
            key = str((p or {}).get("key", "")).lower().strip()
            val = str((p or {}).get("value", "")).strip()
            if key in {"proteinid", "protein id"} and val:
                protein_ids.append(val)

        if db == "refseq":
            refseq_ids.append(xid)
            protein_ids.append(xid)
        elif db == "embl":
            embl_ids.append(xid)
        elif db == "genbank":
            genbank_ids.append(xid)
        elif db == "proteomes":
            proteome_ids.append(xid)
        elif db == "bioproject":
            bioproject_ids.append(xid)
        elif db == "assembly":
            assembly_ids.append(xid)

    return {
        "query_token": query_token,
        "uniprot_accession": str(entry.get("primaryAccession") or "").strip() or None,
        "taxon_id": str(organism_block.get("taxonId") or "").strip() or None,
        "organism": str((organism_block.get("scientificName") or "")).strip() or None,
        "gene_names": _dedupe_keep_order(gene_names),
        "orf_names": _dedupe_keep_order(orf_names),
        "refseq_ids": _dedupe_keep_order(refseq_ids),
        "embl_ids": _dedupe_keep_order(embl_ids),
        "genbank_ids": _dedupe_keep_order(genbank_ids),
        "proteome_ids": _dedupe_keep_order(proteome_ids),
        "bioproject_ids": _dedupe_keep_order(bioproject_ids),
        "assembly_ids": _dedupe_keep_order(assembly_ids),
        "protein_ids": _dedupe_keep_order(protein_ids),
        "sequence_length": int((entry.get("sequence") or {}).get("length") or 0),
    }


def _fetch_uniprot_bundle(accession_id: Optional[str], hit_id: Optional[str]) -> Dict[str, Any]:
    """Try UniProt search for accession/hit token and return a compact hint bundle."""
    tokens = _dedupe_keep_order([accession_id, hit_id])
    for token in tokens:
        data = _safe_uniprot_search(token)
        results = (data or {}).get("results", []) or []
        if results:
            return _extract_uniprot_bundle(results[0], query_token=token)
    return {}


def fetch_nuccore_record(accession: str) -> Optional[Any]:
    """Fetch a nucleotide GenBank record by accession or GI-like id."""
    acc = str(accession or "").strip()
    if not acc:
        return None
    if acc in _cache_nuccore_records:
        return _cache_nuccore_records[acc]

    try:
        time.sleep(NCBI_SLEEP_SEC)
        handle = Entrez.efetch(db="nucleotide", id=acc, rettype="gb", retmode="text")
        rec = SeqIO.read(handle, "genbank")
        handle.close()
        _cache_nuccore_records[acc] = rec
        return rec
    except Exception as exc:
        _warn(f"[WARN] efetch failed for {acc}: {exc}")
        return None


def _extract_taxon_id_from_record(rec: Any) -> Optional[str]:
    try:
        for feat in getattr(rec, "features", []) or []:
            if getattr(feat, "type", "") != "source":
                continue
            for xref in feat.qualifiers.get("db_xref", []) or []:
                sx = str(xref)
                if sx.lower().startswith("taxon:"):
                    return sx.split(":", 1)[1].strip()
    except Exception:
        pass
    return None


def _extract_best_locus(
    rec: Any,
    locus_hints: Sequence[str],
    gene_hints: Sequence[str],
    protein_hints: Optional[Sequence[str]] = None,
) -> Tuple[Optional[str], bool]:
    lhints = set(_dedupe_keep_order(locus_hints))
    ghints = set(_dedupe_keep_order(gene_hints))
    phints = set(_dedupe_keep_order(protein_hints or []))

    first_locus = None
    matched = False
    try:
        for feat in getattr(rec, "features", []) or []:
            if getattr(feat, "type", "") not in {"gene", "CDS"}:
                continue
            qualifiers = getattr(feat, "qualifiers", {}) or {}
            loci = [str(x).strip() for x in qualifiers.get("locus_tag", []) if str(x).strip()]
            genes = [str(x).strip() for x in qualifiers.get("gene", []) if str(x).strip()]
            prots = [str(x).strip() for x in qualifiers.get("protein_id", []) if str(x).strip()]
            if first_locus is None and loci:
                first_locus = loci[0]
            if any(v in lhints for v in loci):
                return loci[0], True
            if any(v in ghints for v in genes):
                matched = matched or bool(loci)
                if loci:
                    return loci[0], True
            if any(v in phints for v in prots):
                matched = matched or bool(loci)
                if loci:
                    return loci[0], True
    except Exception:
        pass
    return first_locus, matched


def _resolve_assembly_from_nuccore(nuccore_id: str) -> Optional[str]:
    """Map nuccore id/accession to assembly accession when possible."""
    nid = str(nuccore_id or "").strip()
    if not nid:
        return None
    try:
        time.sleep(NCBI_SLEEP_SEC)
        handle = Entrez.elink(dbfrom="nuccore", db="assembly", id=nid)
        res = Entrez.read(handle)
        handle.close()
        assembly_ids: List[str] = []
        for ls in res:
            for dbset in ls.get("LinkSetDb", []) or []:
                for link in dbset.get("Link", []) or []:
                    lid = str((link or {}).get("Id", "")).strip()
                    if lid:
                        assembly_ids.append(lid)
        if not assembly_ids:
            return None
        sum_data = _safe_entrez_esummary(db="assembly", ids=assembly_ids[0])
        docs = ((sum_data or {}).get("DocumentSummarySet", {}) or {}).get("DocumentSummary", []) or []
        if docs:
            acc = str((docs[0] or {}).get("AssemblyAccession", "")).strip()
            if acc:
                return acc
    except Exception as exc:
        _warn(f"[WARN] Assembly resolve failed for {nid}: {exc}")
    return None


def resolve_nuccore_from_protein(protein_id: str, retmax: int = 8) -> List[str]:
    """Resolve protein accession to candidate nuccore IDs using NCBI links/search."""
    pid = str(protein_id or "").strip()
    if not pid:
        return []

    if _is_nuccore_like(pid):
        return [pid]

    try:
        time.sleep(NCBI_SLEEP_SEC)
        handle = Entrez.elink(dbfrom="protein", db="nuccore", id=pid)
        res = Entrez.read(handle)
        handle.close()
        candidates: List[str] = []
        for ls in res:
            for dbset in ls.get("LinkSetDb", []):
                for link in dbset.get("Link", []):
                    lid = str(link.get("Id", "")).strip()
                    if lid:
                        candidates.append(lid)
        if candidates:
            return list(dict.fromkeys(candidates))[:retmax]
    except Exception:
        pass

    query = f'"{pid}"[All Fields]'
    es = _safe_entrez_esearch(db="nuccore", term=query, retmax=retmax)
    return [str(x).strip() for x in es.get("IdList", []) if str(x).strip()]


def _score_record(
    rec: Any,
    locus_found: bool,
    genome_accession: Optional[str],
    known_taxon_id: Optional[str],
    hit_taxon_id: Optional[str],
) -> float:
    score = 0.0
    if rec is not None:
        score += 3.5
        bp_len = len(getattr(rec, "seq", "") or "")
        if bp_len >= MIN_RECORD_BP_FOR_EARLY_STOP:
            score += 1.5
    if genome_accession and _is_assembly_like(genome_accession):
        score += 3.0
    elif genome_accession:
        score += 1.0
    if locus_found:
        score += 2.5
    if known_taxon_id and hit_taxon_id and str(known_taxon_id) == str(hit_taxon_id):
        score += 1.0
    elif hit_taxon_id:
        score += 0.4
    return score


def _empty_resolve_result(accession_id: Optional[str]) -> Dict[str, Any]:
    return {
        "resolved_genome_accession": None,
        "resolved_locus_tag": None,
        "resolved_taxon_id": None,
        "locus_found_in_record": False,
        "protein_meta": {
            "best_score": -1.0,
            "best_candidate_source": None,
            "stop_reason": "no_match",
            "input_accession": str(accession_id).strip() if accession_id else None,
        },
        "uniprot_meta": {},
    }


def _candidate_tokens_from_uniprot(bundle: Dict[str, Any]) -> List[str]:
    if not bundle:
        return []
    return _dedupe_keep_order(
        list(bundle.get("refseq_ids", []))
        + list(bundle.get("embl_ids", []))
        + list(bundle.get("genbank_ids", []))
    )


def resolve_record_for_candidate(
    accession_id: Optional[str],
    hit_id: Optional[str],
    fallback_organism: Optional[str],
    known_taxon_id: Optional[str] = None,
) -> Dict[str, Any]:
    """Base resolver used by wrapper cells below."""
    acc = str(accession_id or "").strip()
    hit = str(hit_id or "").strip()
    organism = str(fallback_organism or "").strip()

    result = _empty_resolve_result(accession_id)

    uniprot_bundle = _fetch_uniprot_bundle(acc, hit)
    if uniprot_bundle:
        result["uniprot_meta"] = uniprot_bundle

    taxon_hint = _first_nonempty(known_taxon_id, uniprot_bundle.get("taxon_id") if uniprot_bundle else None)
    organism_hint = _first_nonempty(organism, uniprot_bundle.get("organism") if uniprot_bundle else None)

    locus_hints = _dedupe_keep_order(
        [
            x
            for x in [
                hit if _is_locus_like_tag(hit) else None,
                acc if _is_locus_like_tag(acc) else None,
            ]
            if x
        ]
        + list((uniprot_bundle or {}).get("orf_names", []))
    )
    gene_hints = _dedupe_keep_order(list((uniprot_bundle or {}).get("gene_names", [])))
    protein_hints = _dedupe_keep_order(list((uniprot_bundle or {}).get("protein_ids", [])))

    # New Layer 0: prioritize hit_id as direct genome accession candidate (with .1 variant).
    direct_candidates = _dedupe_keep_order(_nuccore_variants(hit) + _nuccore_variants(acc))

    linked_candidates: List[str] = []
    for token in [acc, hit]:
        if token:
            linked_candidates.extend(resolve_nuccore_from_protein(token, retmax=MAX_NUCCORE_CANDIDATES))
    for token in protein_hints[:8]:
        linked_candidates.extend(resolve_nuccore_from_protein(token, retmax=MAX_CANDIDATES_PER_STAGE))

    uniprot_candidates = _candidate_tokens_from_uniprot(uniprot_bundle)

    candidates = _dedupe_keep_order(direct_candidates + linked_candidates + uniprot_candidates)

    best = None
    for cand in candidates[:MAX_NUCCORE_CANDIDATES]:
        rec = fetch_nuccore_record(cand)
        if rec is None:
            continue
        record_taxon = _extract_taxon_id_from_record(rec)
        locus_tag, locus_found = _extract_best_locus(
            rec,
            locus_hints=locus_hints,
            gene_hints=gene_hints,
            protein_hints=protein_hints,
        )
        rec_id = str(getattr(rec, "id", "")).strip() or cand
        assembly_acc = _resolve_assembly_from_nuccore(rec_id)
        genome_acc = assembly_acc or rec_id
        score = _score_record(
            rec=rec,
            locus_found=locus_found,
            genome_accession=genome_acc,
            known_taxon_id=taxon_hint,
            hit_taxon_id=record_taxon,
        )
        source = "direct_hit_genome" if cand in direct_candidates else "protein_or_uniprot_candidate"
        proposal = {
            "resolved_genome_accession": genome_acc,
            "resolved_locus_tag": locus_tag,
            "resolved_taxon_id": _first_nonempty(record_taxon, taxon_hint),
            "locus_found_in_record": bool(locus_found),
            "score": score,
            "source": source,
        }
        if best is None or proposal["score"] > best["score"]:
            best = proposal
        if proposal["score"] >= EARLY_STOP_SCORE:
            break

    if best is None or best["score"] < ACCEPT_MIN_SCORE:
        text_terms = _dedupe_keep_order(
            list(locus_hints[:8]) + list(gene_hints[:8]) + list(protein_hints[:8]) + ([hit] if hit else []) + ([acc] if acc else [])
        )
        for term in text_terms:
            query_parts = [f'"{term}"[All Fields]']
            if taxon_hint and str(taxon_hint).isdigit():
                query_parts.append(f"txid{str(taxon_hint)}[Organism:exp]")
            if organism_hint:
                query_parts.append(f'"{organism_hint}"[Organism]')
            query = " AND ".join(query_parts)
            es = _safe_entrez_esearch(db="nuccore", term=query, retmax=MAX_CANDIDATES_PER_STAGE)
            ids = [str(x).strip() for x in es.get("IdList", []) if str(x).strip()]
            for cand in ids:
                rec = fetch_nuccore_record(cand)
                if rec is None:
                    continue
                record_taxon = _extract_taxon_id_from_record(rec)
                locus_tag, locus_found = _extract_best_locus(
                    rec,
                    locus_hints=locus_hints,
                    gene_hints=gene_hints,
                    protein_hints=protein_hints,
                )
                rec_id = str(getattr(rec, "id", "")).strip() or cand
                assembly_acc = _resolve_assembly_from_nuccore(rec_id)
                genome_acc = assembly_acc or rec_id
                score = _score_record(
                    rec=rec,
                    locus_found=locus_found,
                    genome_accession=genome_acc,
                    known_taxon_id=taxon_hint,
                    hit_taxon_id=record_taxon,
                )
                proposal = {
                    "resolved_genome_accession": genome_acc,
                    "resolved_locus_tag": locus_tag,
                    "resolved_taxon_id": _first_nonempty(record_taxon, taxon_hint),
                    "locus_found_in_record": bool(locus_found),
                    "score": score,
                    "source": "orf_gene_taxon_text_search",
                }
                if best is None or proposal["score"] > best["score"]:
                    best = proposal
                if proposal["score"] >= EARLY_STOP_SCORE:
                    break
            if best is not None and best["score"] >= EARLY_STOP_SCORE:
                break

    if best is None:
        assembly_ids = list((uniprot_bundle or {}).get("assembly_ids", []))
        if assembly_ids:
            best = {
                "resolved_genome_accession": assembly_ids[0],
                "resolved_locus_tag": None,
                "resolved_taxon_id": taxon_hint,
                "locus_found_in_record": False,
                "score": 7.8,
                "source": "uniprot_assembly_fallback",
            }

    if best is not None:
        result["resolved_genome_accession"] = best.get("resolved_genome_accession")
        result["resolved_locus_tag"] = best.get("resolved_locus_tag")
        result["resolved_taxon_id"] = best.get("resolved_taxon_id")
        result["locus_found_in_record"] = bool(best.get("locus_found_in_record"))
        result["protein_meta"]["best_score"] = float(best.get("score", -1.0))
        result["protein_meta"]["best_candidate_source"] = best.get("source")
        result["protein_meta"]["stop_reason"] = "resolved"
    else:
        result["resolved_taxon_id"] = taxon_hint

    result["protein_meta"]["used_taxon_hint"] = taxon_hint
    result["protein_meta"]["used_organism_hint"] = organism_hint
    result["protein_meta"]["uniprot_hit"] = bool(uniprot_bundle)

    return result


# Stable aliases used by wrappers to avoid recursive re-wrapping on re-execution.
BASE_RESOLVE_RECORD_FOR_CANDIDATE = resolve_record_for_candidate
BASE_RESOLVE_NUCCORE_FROM_PROTEIN = resolve_nuccore_from_protein

In [4]:

# ORF-based search methods and mode-sensitive resolver logic
# ===================================================

def _search_nuccore_by_orf_names(orf_names: Sequence[str], organism: Optional[str], retmax: int = 8) -> List[str]:
    """Search NCBI nuccore by ORF names (locus-like gene identifiers)."""
    org = str(organism).strip() if organism else ""
    out: List[str] = []
    
    for orf in orf_names:
        orf = str(orf).strip()
        if not orf or len(orf) < 2:
            continue
        
        # Try locus-style search
        query = f'"{orf}"[All Fields]'
        if org:
            query += f' AND "{org}"[Organism]'
        try:
            es = _safe_entrez_esearch(db="nuccore", term=query, retmax=retmax)
            out.extend(es.get("IdList", []))
        except Exception as exc:
            _warn(f"[WARN] ORF nuccore search failed for {orf}: {exc}")
    
    return list(dict.fromkeys([str(x).strip() for x in out if str(x).strip()]))


def _search_nuccore_by_gene_names(gene_names: Sequence[str], organism: Optional[str], retmax: int = 8) -> List[str]:
    """Search NCBI nuccore by gene names."""
    org = str(organism).strip() if organism else ""
    out: List[str] = []
    
    for gene in gene_names:
        gene = str(gene).strip()
        if not gene or len(gene) < 2:
            continue
        
        query = f'"{gene}"[All Fields]'
        if org:
            query += f' AND "{org}"[Organism]'
        try:
            es = _safe_entrez_esearch(db="nuccore", term=query, retmax=retmax)
            out.extend(es.get("IdList", []))
        except Exception as exc:
            _warn(f"[WARN] Gene name search failed for {gene}: {exc}")
    
    return list(dict.fromkeys([str(x).strip() for x in out if str(x).strip()]))


def _is_thorough_mode(mode: Optional[str]) -> bool:
    """Check if thorough mode is enabled."""
    m = str(mode or "").strip().lower()
    return m in ("thorough", "all", "exhaustive", "full")


def _is_fast_mode(mode: Optional[str]) -> bool:
    """Check if fast mode is enabled."""
    m = str(mode or "").strip().lower()
    return m in ("fast", "quick", "rapid", "minimal")




In [5]:
# -----------------------------------------------
# Unified resolver wrappers with filters
# -----------------------------------------------

# Always wrap stable base aliases from cell 4 (prevents recursion on re-run)
_base_resolve_record_for_candidate = BASE_RESOLVE_RECORD_FOR_CANDIDATE
_base_resolve_nuccore_from_protein = BASE_RESOLVE_NUCCORE_FROM_PROTEIN


def _best_cds_locus_by_aa_length(rec: Any, target_aa_length: Optional[int]) -> Tuple[Optional[str], Optional[int], Optional[int], Optional[str]]:
    """Pick CDS locus_tag whose translated length best matches UniProt sequence length."""
    try:
        tlen = int(target_aa_length) if target_aa_length is not None else None
    except Exception:
        tlen = None
    if not tlen or rec is None:
        return None, None, None, None

    best = None
    try:
        for feat in getattr(rec, "features", []) or []:
            if getattr(feat, "type", "") != "CDS":
                continue
            q = getattr(feat, "qualifiers", {}) or {}
            loci = [str(x).strip() for x in q.get("locus_tag", []) if str(x).strip()]
            if not loci:
                continue
            trans = [str(x).strip() for x in q.get("translation", []) if str(x).strip()]
            if not trans:
                continue
            aa_len = len(trans[0].replace("\n", "").replace(" ", ""))
            delta = abs(int(aa_len) - int(tlen))
            pid = _first_nonempty(*(q.get("protein_id", []) or [None]))
            cand = (loci[0], int(aa_len), int(delta), pid)
            if best is None or cand[2] < best[2]:
                best = cand
    except Exception:
        return None, None, None, None

    if best is None:
        return None, None, None, None
    return best


def _fetch_uniparc_sequence_length(token: Optional[str]) -> Optional[int]:
    """Fallback: fetch sequence length from UniParc for accessions lacking UniProt sequence data."""
    t = str(token or "").strip()
    if not t:
        return None
    try:
        data = _safe_requests_json(
            "https://rest.uniprot.org/uniparc/search",
            params={"query": t, "format": "json", "size": 1},
            timeout=UNIPROT_TIMEOUT_SEC,
        )
        results = (data or {}).get("results", []) or []
        if not results:
            return None
        seq = (results[0] or {}).get("sequence", {}) or {}
        slen = seq.get("length")
        if slen is None:
            return None
        iv = int(slen)
        return iv if iv > 0 else None
    except Exception:
        return None


def _fetch_uniprot_quick_by_accession(accession_id: Optional[str]) -> Dict[str, Any]:
    """Direct UniProt accession fetch used as fallback when search bundle is empty."""
    acc = str(accession_id or "").strip()
    if not acc or not _looks_uniprot_accession(acc):
        return {}

    try:
        time.sleep(UNIPROT_SLEEP_SEC)
        resp = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=UNIPROT_TIMEOUT_SEC)
        resp.raise_for_status()
        data = resp.json()
    except Exception:
        return {}

    seq_len = int((data.get("sequence") or {}).get("length") or 0)
    if seq_len <= 0:
        up_len = _fetch_uniparc_sequence_length(acc)
        if up_len is not None:
            seq_len = int(up_len)

    organism_block = data.get("organism", {}) or {}
    return {
        "query_token": acc,
        "uniprot_accession": str(data.get("primaryAccession") or acc),
        "taxon_id": str(organism_block.get("taxonId") or "").strip() or None,
        "organism": str(organism_block.get("scientificName") or "").strip() or None,
        "sequence_length": seq_len,
        "gene_names": [],
        "orf_names": [],
        "refseq_ids": [],
        "embl_ids": [],
        "genbank_ids": [],
        "proteome_ids": [],
        "bioproject_ids": [],
        "assembly_ids": [],
    }


def _boost_resolution_score(
    base_score: Optional[float],
    source: Optional[str],
    locus_tag: Optional[str],
    locus_found_in_record: bool,
    aa_delta: Optional[int],
) -> float:
    """Evidence-aware score boost so trustworthy direct hits are not penalized by rigid cutoff."""
    try:
        score = float(base_score) if base_score is not None else -1.0
    except Exception:
        score = -1.0

    src = str(source or "")
    has_locus = bool(_first_nonempty(locus_tag))

    if src in {"direct_hit_genome", "direct_nuccore"}:
        score += 1.0
    if has_locus:
        score += 0.8
    if locus_found_in_record:
        score += 0.8
    if aa_delta is not None:
        if int(aa_delta) == 0:
            score += 1.2
        elif int(aa_delta) <= 5:
            score += 0.9
        elif int(aa_delta) <= 20:
            score += 0.5

    return min(score, 10.0)


# Wrapper 1: Confidence gate + locus correction by AA length
def resolve_record_for_candidate(
    accession_id: Optional[str],
    hit_id: Optional[str],
    fallback_organism: Optional[str],
    known_taxon_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Resolve record with confidence threshold and conservative AA-length post-correction.

    AA-length correction is only used when the base resolver did not provide a robust locus.
    """
    res = _base_resolve_record_for_candidate(
        accession_id=accession_id,
        hit_id=hit_id,
        fallback_organism=fallback_organism,
        known_taxon_id=known_taxon_id,
    )

    pmeta = (res.get("protein_meta", {}) or {}).copy()
    uniprot_meta = (res.get("uniprot_meta", {}) or {}).copy()
    if not uniprot_meta:
        uniprot_meta = _fetch_uniprot_quick_by_accession(accession_id)
        if uniprot_meta:
            res["uniprot_meta"] = uniprot_meta

    target_aa_len = uniprot_meta.get("sequence_length")
    if not target_aa_len:
        target_aa_len = _fetch_uniparc_sequence_length(_first_nonempty(uniprot_meta.get("uniprot_accession"), accession_id))
    best_locus = None
    best_aa_len = None
    best_delta = None
    best_pid = None

    resolved_genome = _first_nonempty(res.get("resolved_genome_accession"))
    current_locus = _first_nonempty(res.get("resolved_locus_tag"))
    locus_found_flag = bool(res.get("locus_found_in_record"))

    # Only attempt AA-based replacement when locus is still weak/missing.
    should_apply_aa_fix = (current_locus is None) or (not locus_found_flag)
    if resolved_genome and target_aa_len and should_apply_aa_fix:
        rec = fetch_nuccore_record(str(resolved_genome))
        best_locus, best_aa_len, best_delta, best_pid = _best_cds_locus_by_aa_length(rec, target_aa_len)
        if best_locus is not None and best_delta is not None and best_delta <= 25:
            res["resolved_locus_tag"] = best_locus
            res["locus_found_in_record"] = True

    if _first_nonempty(res.get("resolved_locus_tag")):
        res["locus_found_in_record"] = True

    boosted_score = _boost_resolution_score(
        base_score=pmeta.get("best_score"),
        source=pmeta.get("best_candidate_source"),
        locus_tag=res.get("resolved_locus_tag"),
        locus_found_in_record=bool(res.get("locus_found_in_record")),
        aa_delta=best_delta,
    )
    pmeta["best_score"] = boosted_score

    score = pmeta.get("best_score")
    locus_found = bool(res.get("locus_found_in_record"))

    accepted = False
    try:
        if score is not None and not pd.isna(score):
            accepted = float(score) >= float(ACCEPT_MIN_SCORE)
            if not locus_found and accepted:
                accepted = float(score) >= (float(ACCEPT_MIN_SCORE) + 1.0)
    except Exception:
        accepted = False

    pmeta["accepted"] = accepted
    pmeta["review_needed"] = not accepted
    if not accepted:
        prev_reason = pmeta.get("stop_reason", "low_conf")
        pmeta["stop_reason"] = f"confidence_filter({prev_reason})"

    pmeta["uniprot_sequence_length"] = target_aa_len
    pmeta["ncbi_cds_aa_length"] = best_aa_len
    pmeta["aa_length_delta"] = best_delta
    pmeta["best_cds_protein_id"] = best_pid
    pmeta["curated_locus_override_key"] = None

    res["protein_meta"] = pmeta
    return res


# Wrapper 2: Keep protein resolver available for direct calls
def resolve_nuccore_from_protein(protein_id: str, retmax: int = 8) -> List[str]:
    """Delegate to base protein->nuccore resolver."""
    return _base_resolve_nuccore_from_protein(protein_id, retmax=retmax)


print("Unified resolver active: evidence-aware scoring + automatic hit_id genome layer (no curated overrides).")

Unified resolver active: evidence-aware scoring + automatic hit_id genome layer (no curated overrides).


In [6]:

# Extended resolver with ORF-name and gene-name searches
# ======================================================

def resolve_record_for_candidate_extended(
    accession_id: Optional[str],
    hit_id: Optional[str],
    fallback_organism: Optional[str],
    known_taxon_id: Optional[str] = None,
    mode: str = "balanced",
    skip_orf_search: bool = False,
    skip_text_search: bool = False,
) -> Dict[str, Any]:
    """
    Extended resolver with ORF/gene-name searches and mode control.
    
    Modes:
    - 'fast': Stages 1-3 only
    - 'balanced': Stages 1-5a (+ ORF/gene search)
    - 'thorough': All stages including expensive text searches
    """
    # First get base resolution
    res = resolve_record_for_candidate(
        accession_id=accession_id,
        hit_id=hit_id,
        fallback_organism=fallback_organism,
        known_taxon_id=known_taxon_id,
    )
    
    # Mode-aware attribute
    pmeta = (res.get("protein_meta", {}) or {}).copy()
    pmeta["resolver_mode"] = mode
    pmeta["extended"] = True
    
    # If already high confidence or in fast mode, return early
    score = pmeta.get("best_score", -1)
    if score >= EARLY_STOP_SCORE or _is_fast_mode(mode):
        res["protein_meta"] = pmeta
        return res
    
    # Extract ORF and gene names for additional searches
    uniprot_bundle = res.get("uniprot_meta", {}) or {}
    orf_names = uniprot_bundle.get("orf_names", [])
    gene_names = uniprot_bundle.get("gene_names", [])
    
    # Try ORF-based search if not already found
    if (not skip_orf_search) and orf_names and score < EARLY_STOP_SCORE:
        orf_results = _search_nuccore_by_orf_names(orf_names, fallback_organism, retmax=10)
        if orf_results and not res.get("resolved_genome_accession"):
            # Try first ORF result
            try:
                rec = fetch_nuccore_record(orf_results[0])
                if rec:
                    pmeta["extended_orf_search"] = orf_results[0]
                    res["resolved_genome_accession"] = rec.id
                    pmeta["best_candidate_source"] = "extended_orf_search"
            except Exception as exc:
                _warn(f"[WARN] Extended ORF search fetch failed: {exc}")
    
    # Try gene-name search if still unresolved
    if (not skip_text_search) and gene_names and not res.get("resolved_genome_accession"):
        gene_results = _search_nuccore_by_gene_names(gene_names, fallback_organism, retmax=8)
        if gene_results:
            try:
                rec = fetch_nuccore_record(gene_results[0])
                if rec:
                    pmeta["extended_gene_search"] = gene_results[0]
                    res["resolved_genome_accession"] = rec.id
                    pmeta["best_candidate_source"] = "extended_gene_search"
            except Exception as exc:
                _warn(f"[WARN] Extended gene search fetch failed: {exc}")
    
    res["protein_meta"] = pmeta
    return res


print("Extended resolver with ORF/gene-name search active.")


Extended resolver with ORF/gene-name search active.


In [7]:
# ====================================================
# Optional: Advanced configuration and monitoring
# ====================================================

# These classes provide advanced configuration options for power users
# who want fine-grained control over resolver behavior and performance tracking.
# For typical usage, the default resolve_record_for_candidate wrapper is sufficient.


class ResolverConfig:
    """Configuration for resolver behavior."""
    
    def __init__(self,
                 mode: str = "balanced",
                 skip_orf_search: bool = False,
                 skip_text_search: bool = False,
                 enable_uniparc_fallback: bool = True,
                 cache_results: bool = True,
                 max_orf_search_terms: int = 5,
                 max_gene_search_terms: int = 5):
        self.mode = mode
        self.skip_orf_search = skip_orf_search
        self.skip_text_search = skip_text_search
        self.enable_uniparc_fallback = enable_uniparc_fallback
        self.cache_results = cache_results
        self.max_orf_search_terms = max_orf_search_terms
        self.max_gene_search_terms = max_gene_search_terms
    
    def is_fast(self) -> bool:
        return self.mode.lower() in ("fast", "quick", "rapid")
    
    def is_thorough(self) -> bool:
        return self.mode.lower() in ("thorough", "full", "exhaustive")


class ResolverPerformanceMonitor:
    """Track resolver performance metrics."""
    
    def __init__(self):
        self.calls = 0
        self.total_time = 0.0
        self.successes = 0
        self.by_mode = {}
        self.by_source = {}
    
    def record_call(self, mode: str, elapsed: float, success: bool, source: Optional[str] = None):
        """Record resolver call metrics."""
        self.calls += 1
        self.total_time += elapsed
        
        if success:
            self.successes += 1
        
        if mode not in self.by_mode:
            self.by_mode[mode] = {"count": 0, "time": 0.0, "successes": 0}
        self.by_mode[mode]["count"] += 1
        self.by_mode[mode]["time"] += elapsed
        if success:
            self.by_mode[mode]["successes"] += 1
        
        if source:
            if source not in self.by_source:
                self.by_source[source] = {"count": 0, "time": 0.0}
            self.by_source[source]["count"] += 1
            self.by_source[source]["time"] += elapsed
    
    def get_summary(self) -> Dict[str, Any]:
        """Get performance summary."""
        avg_time = self.total_time / max(1, self.calls)
        success_rate = self.successes / max(1, self.calls)
        
        return {
            "total_calls": self.calls,
            "total_time": self.total_time,
            "average_time": avg_time,
            "success_rate": success_rate,
            "by_mode": self.by_mode,
            "by_source": self.by_source,
        }
    
    def __repr__(self) -> str:
        s = self.get_summary()
        lines = [
            f"Resolver Performance Monitor:",
            f"  Total calls: {s['total_calls']}",
            f"  Success rate: {s['success_rate']:.1%}",
            f"  Average time: {s['average_time']:.3f}s",
            f"  Total time: {s['total_time']:.1f}s",
        ]
        if s["by_mode"]:
            lines.append("  By mode:")
            for mode, stats in s["by_mode"].items():
                lines.append(f"    {mode}: {stats['count']} calls, {stats['time']:.2f}s, {stats['successes']} success")
        return "\n".join(lines)


# Global instance for optional performance tracking
RESOLVER_PERF_MONITOR = ResolverPerformanceMonitor()

# Default config
DEFAULT_RESOLVER_CONFIG = ResolverConfig(mode=RESOLUTION_MODE)

print(f"Optional advanced config initialized (default mode: {RESOLUTION_MODE}).")


Optional advanced config initialized (default mode: standard).


In [8]:
# ====================================================
# Optional: Merge, sort, and deduplication logic
# ====================================================

def merge_with_template(new_df: pd.DataFrame, template_df: pd.DataFrame, append_to_template: bool = True) -> pd.DataFrame:
    """
    Merge new rows with template, preserving original order and handling duplicates.
    
    Strategy:
    1. Keep template rows in original order (no sorting/dedup of template)
    2. Remove from new_df any rows that would duplicate template rows
    3. Rank remaining new rows by quality (curated > locus_found > reviewed > score > bp_length)
    4. Concat template (unchanged) + ranked new rows
    5. Final sort by SSN_x_cluster_y for consistent ordering
    """
    template_df = template_df.copy()

    merged_cols = list(template_df.columns)
    for c in new_df.columns:
        if c not in merged_cols:
            merged_cols.append(c)

    template_df = _ensure_columns(template_df, merged_cols)
    new_df = _ensure_columns(new_df, merged_cols)

    if not append_to_template:
        # Not appending: just use new data, rank and sort by SSN
        new_df = _rank_rows_for_dedupe(new_df)
        new_df = new_df.sort_values(
            by=["__rank_curated", "__rank_locus", "__rank_review", "__rank_score", "__rank_bp"],
            ascending=[False, False, False, False, False],
        ).reset_index(drop=True)
        new_df = new_df.drop(columns=[c for c in new_df.columns if c.startswith("__rank_")], errors="ignore")
        new_df = _sort_by_ssn_cluster(new_df)
        return new_df
    
    # Appending mode: preserve template order, dedupe new rows from template
    dedupe_keys = [c for c in DEDUPE_KEY_COLUMNS if c in template_df.columns]
    
    # Track template rows by dedup key
    template_dedup_sigs = set()
    for _, row in template_df.iterrows():
        if dedupe_keys:
            sig = tuple(str(_first_nonempty(row.get(c)) or "") for c in dedupe_keys)
            template_dedup_sigs.add(sig)
    
    # Filter new_df to remove duplicates of template rows
    new_df_filtered = []
    for _, row in new_df.iterrows():
        if dedupe_keys:
            sig = tuple(str(_first_nonempty(row.get(c)) or "") for c in dedupe_keys)
            if sig in template_dedup_sigs:
                continue
        new_df_filtered.append(row)
    
    if new_df_filtered:
        new_df = pd.DataFrame(new_df_filtered).reset_index(drop=True)
    else:
        new_df = pd.DataFrame()
    
    # Rank and sort new rows only
    if not new_df.empty:
        new_df = _rank_rows_for_dedupe(new_df)
        new_df = new_df.sort_values(
            by=["__rank_curated", "__rank_locus", "__rank_review", "__rank_score", "__rank_bp"],
            ascending=[False, False, False, False, False],
        ).reset_index(drop=True)
        new_df = new_df.drop(columns=[c for c in new_df.columns if c.startswith("__rank_")], errors="ignore")
    
    # Concat: template first (original order), then new rows
    if new_df.empty:
        out = template_df.copy()
    else:
        out = pd.concat([template_df, new_df], ignore_index=True)
    
    # Final sort by SSN and cluster numerically
    out = _sort_by_ssn_cluster(out)
    return out


def _sort_by_ssn_cluster(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort dataframe by SSN ID and cluster ID numerically.
    Ensures ssn_1_cluster_1 comes before ssn_1_cluster_2, then ssn_2_cluster_1, etc.
    """
    if df.empty or "Source SSN ID" not in df.columns or "Source Cluster ID" not in df.columns:
        return df.reset_index(drop=True)
    
    df = df.copy()
    df["__ssn_num"] = pd.to_numeric(df["Source SSN ID"], errors="coerce").fillna(-1).astype(int)
    df["__cluster_num"] = pd.to_numeric(df["Source Cluster ID"], errors="coerce").fillna(-1).astype(int)
    
    df = df.sort_values(
        by=["__ssn_num", "__cluster_num"],
        ascending=[True, True],
        na_position="last"
    ).reset_index(drop=True)
    
    df = df.drop(columns=["__ssn_num", "__cluster_num"], errors="ignore")
    return df


print("Merge and deduplication functions active.")


Merge and deduplication functions active.


In [9]:
# ======================================================================
# BUILD PIPELINE: Run resolver on selected candidates and create output
# ======================================================================

print("=" * 70)
print("RUNNING RESOLUTION PIPELINE ON SELECTED CANDIDATES")
print("=" * 70)

# Load template if appending
template_df = None
if APPEND_TO_TEMPLATE and os.path.exists(TEMPLATE_CSV_PATH):
    try:
        template_df = pd.read_csv(TEMPLATE_CSV_PATH, sep=OUTPUT_CSV_SEP)
        print(f"\nLoaded template: {len(template_df)} rows from {os.path.basename(TEMPLATE_CSV_PATH)}")
    except Exception as e:
        print(f"[WARN] Failed to load template: {e}")
        template_df = None

# Start with selected representatives from clustering
new_rows = []

if "selected_df" not in globals() or selected_df is None or selected_df.empty:
    print("[WARN] No selected_df available; skipping resolution pipeline")
    new_rows_df = pd.DataFrame()
else:
    print(f"\nResolving {len(selected_df)} selected candidates...")

    mode_map = {"fast": "fast", "standard": "balanced", "depth": "thorough"}
    ext_mode = mode_map.get(str(RESOLUTION_MODE).strip().lower(), "balanced")
    use_extended = "resolve_record_for_candidate_extended" in globals()

    for idx, row in selected_df.iterrows():
        accession_id = row.get("accession_id", row.get("Accession"))
        hit_id = row.get("hit_id", row.get("Hit ID", row.get("Protein ID")))
        organism = row.get("organism", row.get("Organism", ""))
        known_taxon = _first_nonempty(row.get("taxon_id"), row.get("Taxon ID"), row.get("taxonomy_id"))

        if use_extended:
            result = resolve_record_for_candidate_extended(
                accession_id=accession_id,
                hit_id=hit_id,
                fallback_organism=organism,
                known_taxon_id=known_taxon,
                mode=ext_mode,
            )
        else:
            result = resolve_record_for_candidate(
                accession_id=accession_id,
                hit_id=hit_id,
                fallback_organism=organism,
                known_taxon_id=known_taxon,
            )

        resolved_row = dict(row)

        # Normalize key output/template fields so appended rows are not empty in target columns.
        resolved_row["Enzyme"] = _first_nonempty(row.get("Enzyme"), row.get("neighborhood_label"), row.get("hit_id"))
        resolved_row["Organism"] = _first_nonempty(row.get("Organism"), row.get("organism"))
        resolved_row["Protein ID"] = _first_nonempty(row.get("Protein ID"), row.get("hit_id"), row.get("accession_id"))

        uniprot_meta = result.get("uniprot_meta") or {}
        pmeta = result.get("protein_meta") or {}

        resolved_row["Locus Tag"] = _first_nonempty(result.get("resolved_locus_tag"), row.get("Locus Tag"))
        resolved_row["Genome Accession"] = _first_nonempty(
            result.get("resolved_genome_accession"),
            (uniprot_meta.get("assembly_ids") or [None])[0],
            row.get("Genome Accession"),
        )
        resolved_row["Genebank ID"] = _first_nonempty(
            result.get("resolved_genome_accession"),
            (uniprot_meta.get("refseq_ids") or [None])[0],
            (uniprot_meta.get("embl_ids") or [None])[0],
            row.get("accession_id"),
        )
        resolved_row["Taxon ID"] = _first_nonempty(
            result.get("resolved_taxon_id"),
            uniprot_meta.get("taxon_id"),
            row.get("Taxon ID"),
            row.get("taxon_id"),
            row.get("taxonomy_id"),
        )
        resolved_row["Locus Found In Record"] = result.get("locus_found_in_record")

        # Keep source traceability for downstream sorting/inspection.
        resolved_row["Source SSN ID"] = _first_nonempty(row.get("Source SSN ID"), row.get("ssn_id"))
        resolved_row["Source Cluster ID"] = _first_nonempty(row.get("Source Cluster ID"), row.get("cluster_id"))
        resolved_row["Source Neighborhood Label"] = _first_nonempty(row.get("Source Neighborhood Label"), row.get("neighborhood_label"))

        # Resolver diagnostics and UniProt-derived metadata
        resolved_row["Resolution Score"] = pmeta.get("best_score", -1.0)
        resolved_row["Resolution Source"] = pmeta.get("best_candidate_source")
        resolved_row["Stop Reason"] = pmeta.get("stop_reason")
        resolved_row["Review Needed"] = bool(pmeta.get("review_needed", False))
        resolved_row["UniProt Query Token"] = uniprot_meta.get("query_token")
        resolved_row["UniProt Accession"] = uniprot_meta.get("uniprot_accession")
        resolved_row["UniProt ORF Names"] = "; ".join(uniprot_meta.get("orf_names", []))
        resolved_row["UniProt Gene Names"] = "; ".join(uniprot_meta.get("gene_names", []))
        resolved_row["UniProt Proteome IDs"] = "; ".join(uniprot_meta.get("proteome_ids", []))
        resolved_row["UniProt BioProject IDs"] = "; ".join(uniprot_meta.get("bioproject_ids", []))
        resolved_row["UniProt RefSeq IDs"] = "; ".join(uniprot_meta.get("refseq_ids", []))
        resolved_row["UniProt EMBL IDs"] = "; ".join(uniprot_meta.get("embl_ids", []))
        resolved_row["UniProt Sequence Length"] = pmeta.get("uniprot_sequence_length")
        resolved_row["NCBI CDS AA Length"] = pmeta.get("ncbi_cds_aa_length")
        resolved_row["AA Length Delta"] = pmeta.get("aa_length_delta")
        resolved_row["NCBI CDS Protein ID"] = pmeta.get("best_cds_protein_id")
        resolved_row["Curated Locus Override Key"] = pmeta.get("curated_locus_override_key")

        new_rows.append(resolved_row)
        if (idx + 1) % max(1, len(selected_df) // 10) == 0:
            print(f"  Processed {idx + 1}/{len(selected_df)} rows...")

    new_rows_df = pd.DataFrame(new_rows)
    print(f"\nResolution complete: {len(new_rows_df)} rows processed")

# Upsert matching template rows (no duplicates) and keep only truly new rows for appending.
if template_df is not None and not new_rows_df.empty:
    def _norm_key(v: Any) -> str:
        vv = _first_nonempty(v)
        return "" if vv is None else str(vv).strip().upper()

    template_df = template_df.copy()
    for c in new_rows_df.columns:
        if c not in template_df.columns:
            template_df[c] = None

    # Order matters: prefer biologically strong identifiers first.
    match_specs = [
        ("Genebank ID", "genebank"),
        ("Genome Accession", "genome"),
        ("Locus Tag", "locus"),
        ("Enzyme", "enzyme"),
    ]

    prefer_new_cols = {
        "Genome Accession", "Genebank ID", "Locus Tag", "Organism", "Taxon ID", "Protein ID",
        "Locus Found In Record", "Resolution Score", "Resolution Source", "Stop Reason", "Review Needed",
        "UniProt Query Token", "UniProt Accession", "UniProt ORF Names", "UniProt Gene Names",
        "UniProt Proteome IDs", "UniProt BioProject IDs", "UniProt RefSeq IDs", "UniProt EMBL IDs",
        "UniProt Sequence Length", "NCBI CDS AA Length", "AA Length Delta", "NCBI CDS Protein ID",
        "Curated Locus Override Key", "Source SSN ID", "Source Cluster ID", "Source Neighborhood Label",
    }

    rows_for_append = []
    updated_count = 0
    for _, nrow in new_rows_df.iterrows():
        matched_idx = None
        matched_reason = None

        for col, reason in match_specs:
            if col not in template_df.columns or col not in nrow.index:
                continue
            kval = _norm_key(nrow.get(col))
            if not kval:
                continue
            hit_idx = template_df.index[template_df[col].map(_norm_key) == kval].tolist()
            if len(hit_idx) == 1:
                matched_idx = hit_idx[0]
                matched_reason = reason
                break

        if matched_idx is None:
            rows_for_append.append(nrow.to_dict())
            continue

        updated_count += 1
        for col in template_df.columns:
            if col not in nrow.index:
                continue
            new_val = nrow.get(col)
            if _first_nonempty(new_val) is None:
                continue

            old_val = template_df.at[matched_idx, col]
            old_empty = _first_nonempty(old_val) is None

            if old_empty or col in prefer_new_cols:
                template_df.at[matched_idx, col] = new_val

        if "Review Needed" in template_df.columns:
            template_df.at[matched_idx, "Review Needed"] = False

    if updated_count > 0:
        print(f"Updated {updated_count} template rows from generated matches (no duplicates appended).")

    new_rows_df = pd.DataFrame(rows_for_append) if rows_for_append else pd.DataFrame(columns=new_rows_df.columns)
    if len(new_rows_df) > 0:
        print(f"Appending {len(new_rows_df)} truly new rows.")

# Merge with template if applicable
if template_df is not None and not new_rows_df.empty:
    print("\nMerging with template...")
    if "merge_with_template" in globals():
        final_df = merge_with_template(new_rows_df, template_df, append_to_template=APPEND_TO_TEMPLATE)
    else:
        print("[WARN] merge_with_template not loaded yet; using concat fallback.")
        final_df = pd.concat([template_df, new_rows_df], ignore_index=True)
elif template_df is not None and (new_rows_df.empty or len(new_rows_df) == 0):
    print("\nNo additional new rows to append; using updated template as final dataset...")
    final_df = template_df.copy()
elif not new_rows_df.empty:
    print("\nNo template; using resolved rows as final dataset...")
    final_df = new_rows_df.copy()
else:
    print("\nNo resolved rows available; creating empty dataset...")
    final_df = pd.DataFrame()

# Ensure required columns without dropping existing columns
if not final_df.empty:
    if "_ensure_columns" in globals():
        final_df = _ensure_columns(final_df, REQUIRED_OUTPUT_COLUMNS)
    else:
        for c in REQUIRED_OUTPUT_COLUMNS:
            if c not in final_df.columns:
                final_df[c] = None

    # Coalesce case-variant columns into canonical names for clean CSV headers.
    canonical_pairs = [
        ("Organism", "organism"),
        ("Protein ID", "hit_id"),
        ("Source SSN ID", "ssn_id"),
        ("Source Cluster ID", "cluster_id"),
        ("Taxon ID", "taxon_id"),
    ]
    for canon, alias in canonical_pairs:
        if canon in final_df.columns and alias in final_df.columns:
            final_df[canon] = final_df[canon].where(final_df[canon].notna(), final_df[alias])
            final_df[canon] = final_df[canon].replace("", pd.NA).fillna(final_df[alias])
            final_df = final_df.drop(columns=[alias], errors="ignore")

    # Remove any accidental duplicate column names while keeping first occurrence.
    final_df = final_df.loc[:, ~final_df.columns.duplicated(keep="first")]

    final_df.to_csv(OUTPUT_CSV_PATH, sep=OUTPUT_CSV_SEP, index=False)
    print(f"\nSaved output: {len(final_df)} rows -> {os.path.basename(OUTPUT_CSV_PATH)}")
    print(f"Columns in output: {len(final_df.columns)}")
else:
    print("[WARN] Final dataset is empty; skipping save")

print("=" * 70)

RUNNING RESOLUTION PIPELINE ON SELECTED CANDIDATES

Loaded template: 11 rows from enzyme_ids.csv

Resolving 6 selected candidates...
  Processed 1/6 rows...
  Processed 2/6 rows...
  Processed 3/6 rows...
  Processed 4/6 rows...
  Processed 5/6 rows...
  Processed 6/6 rows...

Resolution complete: 6 rows processed
Updated 1 template rows from generated matches (no duplicates appended).
Appending 5 truly new rows.

Merging with template...

Saved output: 16 rows -> enzyme_ids_augmented_from_gnn.csv
Columns in output: 48


## Quality Control (QC)

This section validates generated rows from the enrichment pipeline and creates a compact QC summary.

Checks include:
- Required fields present (`Locus Tag`, `Genome Accession`, `Organism`, `Taxon ID`)
- Taxon ID is numeric
- `Genome Accession` and `Genebank ID` look plausible
- `Locus Found In Record` consistency (if present)

A CSV report is written to `enzyme_ids_augmented_from_gnn_qc_report.csv`.

In [10]:
# ----------------------------------------------------------------------
# QC: validate generated rows and export a report
# ----------------------------------------------------------------------

QC_REPORT_PATH = os.path.join(BASE_DIR, "enzyme_ids_augmented_from_gnn_qc_report.csv")


def _is_nonempty(v: Any) -> bool:
    if v is None:
        return False
    try:
        if pd.isna(v):
            return False
    except Exception:
        pass
    s = str(v).strip()
    return s != "" and s.lower() not in {"nan", "none", "null", "<na>"}


def _looks_numeric(v: Any) -> bool:
    if not _is_nonempty(v):
        return False
    return str(v).strip().isdigit()


def _looks_assembly_accession(v: Any) -> bool:
    if not _is_nonempty(v):
        return False
    return bool(re.match(r"^(GCA|GCF)_\d+(\.\d+)?$", str(v).strip(), flags=re.IGNORECASE))


def _looks_nuccore_accession(v: Any) -> bool:
    if not _is_nonempty(v):
        return False
    s = str(v).strip()
    patterns = [
        r"^[A-Z]{1,4}_[A-Z0-9]+(\.\d+)?$",      # NC_019940.1, NZ_BJTG01000010.1
        r"^[A-Z]{1,2}\d{5,8}(\.\d+)?$",         # MT631715.1, CP003051
        r"^[A-Z]{4}\d{8}(\.\d+)?$",             # DRQG01000086.1
        r"^[A-Z]{5,6}\d{6,12}(\.\d+)?$",        # JBMORJ000000000.1, JABFGF010000001.1
    ]
    return any(bool(re.match(p, s, flags=re.IGNORECASE)) for p in patterns)


def _is_generated_row(r: pd.Series) -> bool:
    source_fields = ["Source SSN ID", "Source Cluster ID", "Source Neighborhood Label"]
    return any(_is_nonempty(r.get(c)) for c in source_fields)


def _looks_locus_like(v: Any) -> bool:
    if not _is_nonempty(v):
        return False
    s = str(v).strip()
    # Allow '-' in the suffix for metagenome-style locus tags.
    return bool(re.match(r"^[A-Za-z][A-Za-z0-9]{1,24}_[A-Za-z0-9][A-Za-z0-9-]{1,}$", s))


def _resolution_score_ok(r: pd.Series) -> bool:
    """Evidence-aware score gate replacing rigid >=8 cutoff."""
    score = r.get("Resolution Score")
    try:
        if score is not None and not pd.isna(score) and float(score) >= 7.0:
            return True
    except Exception:
        pass

    source = str(r.get("Resolution Source") or "")
    has_locus = _is_nonempty(r.get("Locus Tag"))
    tax_ok = _looks_numeric(r.get("Taxon ID"))
    aa_delta = r.get("AA Length Delta")
    aa_match = False
    try:
        aa_match = aa_delta is not None and not pd.isna(aa_delta) and float(aa_delta) <= 10
    except Exception:
        aa_match = False

    if source == "direct_nuccore" and has_locus and tax_ok:
        return True
    if has_locus and tax_ok and aa_match:
        return True
    return False


def build_qc_report(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    generated = df[df.apply(_is_generated_row, axis=1)].copy()

    rows = []
    for _, r in generated.iterrows():
        locus = r.get("Locus Tag")
        genome = r.get("Genome Accession")
        organism = r.get("Organism")
        taxon = r.get("Taxon ID")
        genebank = r.get("Genebank ID")
        locus_found = r.get("Locus Found In Record")
        score = r.get("Resolution Score")
        master_wgs = r.get("Master WGS Candidate")

        checks = {
            "required_locus_present": _is_nonempty(locus),
            "required_genome_present": _is_nonempty(genome),
            "required_organism_present": _is_nonempty(organism),
            "required_taxon_present": _is_nonempty(taxon),
            "taxon_numeric": _looks_numeric(taxon),
            "locus_tag_plausible": _looks_locus_like(locus),
            "genome_accession_plausible": _looks_assembly_accession(genome) or _looks_nuccore_accession(genome),
            "genebank_id_plausible": (not _is_nonempty(genebank)) or _looks_nuccore_accession(genebank),
            "locus_found_or_missing_flag": (not _is_nonempty(locus_found)) or str(locus_found).strip().lower() in {"true", "1"},
            "not_master_wgs_only_hit": str(master_wgs).strip().lower() not in {"true", "1"},
        }

        core_checks = [
            checks["required_locus_present"],
            checks["required_genome_present"],
            checks["required_organism_present"],
            checks["required_taxon_present"],
            checks["taxon_numeric"],
            checks["locus_tag_plausible"],
            checks["genome_accession_plausible"],
        ]
        qc_pass = all(core_checks)

        score_ok = _resolution_score_ok(r)
        if not score_ok:
            qc_pass = False

        n_ok = int(sum(1 for ok in checks.values() if ok))
        max_score = len(checks) + 1  # +1 for resolution score/evidence gate
        total_score = n_ok + (1 if score_ok else 0)

        rows.append(
            {
                "Enzyme": r.get("Enzyme"),
                "Organism": organism,
                "Locus Tag": locus,
                "Genome Accession": genome,
                "Genebank ID": genebank,
                "Taxon ID": taxon,
                "Source SSN ID": r.get("Source SSN ID"),
                "Source Cluster ID": r.get("Source Cluster ID"),
                "Resolution Score": score,
                "resolution_score_ge_8": score_ok,
                "QC Score": total_score,
                "QC Max Score": max_score,
                "QC Pass": qc_pass,
                **checks,
            }
        )

    qc_df = pd.DataFrame(rows)
    if qc_df.empty:
        return qc_df
    return qc_df.sort_values(by=["QC Pass", "QC Score"], ascending=[True, True]).reset_index(drop=True)


if "final_df" not in globals() or final_df is None or final_df.empty:
    print("[INFO] final_df is not available. Run the build/export cells first.")
else:
    qc_df = build_qc_report(final_df)
    qc_df.to_csv(QC_REPORT_PATH, sep=OUTPUT_CSV_SEP, index=False)

    total = len(qc_df)
    passed = int(qc_df["QC Pass"].sum()) if total > 0 else 0
    failed = total - passed

    print("QC report saved:", QC_REPORT_PATH)
    print(f"QC summary (generated rows): total={total}, passed={passed}, failed={failed}")

    if failed > 0:
        print("\nRows that failed QC:")
        display(
            qc_df.loc[~qc_df["QC Pass"], [
                "Enzyme", "Organism", "Locus Tag", "Genome Accession", "Genebank ID", "Taxon ID",
                "required_locus_present", "required_genome_present", "required_organism_present",
                "required_taxon_present", "taxon_numeric", "locus_tag_plausible",
                "genome_accession_plausible", "genebank_id_plausible", "locus_found_or_missing_flag",
                "not_master_wgs_only_hit", "resolution_score_ge_8", "QC Score", "QC Max Score"
            ]]
        )
    else:
        print("All generated rows passed QC.")

    display(qc_df.head(12))

QC report saved: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\enzyme_ids_augmented_from_gnn_qc_report.csv
QC summary (generated rows): total=6, passed=6, failed=0
All generated rows passed QC.


,Enzyme,Organism,Locus Tag,Genome Accession,Genebank ID,Taxon ID,Source SSN ID,Source Cluster ID,Resolution Score,resolution_score_ge_8,...,required_locus_present,required_genome_present,required_organism_present,required_taxon_present,taxon_numeric,locus_tag_plausible,genome_accession_plausible,genebank_id_plausible,locus_found_or_missing_flag,not_master_wgs_only_hit
0,Thiohalomonas denitrificans._FMWD01000006,Thiohalomonas denitrificans.,SAMN03097708_02041,GCF_900102855.1,GCF_900102855.1,415747,1,1,10.0,True,...,True,True,True,True,True,True,True,True,True,True
1,Anaeromyxobacter diazotrophicus._BJTG01000010,Anaeromyxobacter diazotrophicus.,AMYX_37370,GCF_013340205.1,GCF_013340205.1,2590199,1,2,10.0,True,...,True,True,True,True,True,True,True,True,True,True
2,bacterium BMS3Abin03._BDSV01000107,bacterium BMS3Abin03.,BMS3Abin03_01317,GCA_002898075.1,GCA_002898075.1,2005711,1,3,9.5,True,...,True,True,True,True,True,True,True,True,True,True
3,Thioflavicoccus mobilis 8321._CP003051,Thioflavicoccus mobilis 8321.,Thimo_0001,GCF_000327045.1,GCF_000327045.1,765912,1,4,10.0,True,...,True,True,True,True,True,True,True,True,True,True
4,MeMES,Candidatus Methanophaga sp. ANME-1 ERB7.,ACBHHCEK_00006,MT631715.1,MT631715.1,2759913,1,5,8.7,True,...,True,True,True,True,True,True,True,True,True,True
5,Thermoprotei archaeon._DSCF01000035,Thermoprotei archaeon.,ENN59_07320,DSCF01000035.1,DSCF01000035.1,2250277,3,6,10.0,True,...,True,True,True,True,True,True,True,True,True,True
